# SciFact retrieval + claim verification: full pipeline (Inter-IIT NLP bootcamp)

1. **Runtime → Change runtime type → T4 GPU**.
2. Fill in your name / roll number in the next cell.
3. **Runtime → Run all**. The data cell asks you to upload the provided data as a zip (a zip of the `data/` folder with `corpus.jsonl`, `*_claims.jsonl`), or set `DRIVE_DATA_DIR` to a Google Drive folder that contains those files.
4. The last cell checks the format and downloads `<rollno>_nlp_bootcamp.zip`.

Expected time on a T4: about 35–50 minutes.

In [ ]:
NAME = "Shivansh"  # your name
ROLL_NO = "24b1534"  # your roll number (used for the zip file name)
DRIVE_DATA_DIR = ""  # e.g. '/content/drive/MyDrive/nlp_bootcamp/data' (leave empty to upload a zip)

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip -q install PyStemmer sentence-transformers transformers sentencepiece scikit-learn

In [ ]:
import os, glob, shutil, zipfile
os.makedirs('/content/submission/code', exist_ok=True)
if not os.path.exists('/content/data/corpus.jsonl'):
    if DRIVE_DATA_DIR:
        from google.colab import drive
        drive.mount('/content/drive')
        shutil.copytree(DRIVE_DATA_DIR, '/content/data', dirs_exist_ok=True)
    else:
        from google.colab import files
        up = files.upload()   # choose the data zip
        for name in up:
            if name.endswith('.zip'):
                zipfile.ZipFile(name).extractall('/content/_data_raw')
            else:
                os.makedirs('/content/_data_raw', exist_ok=True)
                shutil.move(name, '/content/_data_raw/' + name)
        src = os.path.dirname(glob.glob('/content/_data_raw/**/corpus.jsonl', recursive=True)[0])
        shutil.copytree(src, '/content/data', dirs_exist_ok=True)
print(sorted(os.listdir('/content/data')))

## Code files (written to `/content/submission/code/`)

In [ ]:
%%writefile /content/submission/code/common.py
"""Shared helpers: data loading, TREC I/O, metrics, timing."""
import json
import math
import os
import random

import numpy as np

SEED = 42
TOP_K = 100

HERE = os.path.dirname(os.path.abspath(__file__))
ROOT = os.path.dirname(HERE)                      # submission/
DATA_DIR = os.environ.get("DATA_DIR", os.path.join(os.path.dirname(ROOT), "data"))
PRED_DIR = os.path.join(ROOT, "predictions")
CACHE_DIR = os.path.join(ROOT, "cache")           # not shipped in the zip
os.makedirs(PRED_DIR, exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)


def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    try:
        import torch
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    except ImportError:
        pass


def read_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]


def load_corpus():
    docs = read_jsonl(os.path.join(DATA_DIR, "corpus.jsonl"))
    doc_ids = [int(d["doc_id"]) for d in docs]
    texts = [d["title"] + " " + " ".join(d["abstract"]) for d in docs]
    return docs, doc_ids, texts


def load_claims(split):
    return read_jsonl(os.path.join(DATA_DIR, f"{split}_claims.jsonl"))


def qrels_of(claims):
    """{claim_id: {doc_id: 1}} for claims with evidence (NEI claims are skipped, as in grading)."""
    q = {}
    for c in claims:
        ev = c.get("evidence") or []
        if ev:
            q[int(c["id"])] = {int(e["doc_id"]): 1 for e in ev}
    return q


# ---------------------------------------------------------------- runs
def topk_from_scores(score_matrix, doc_ids, k=TOP_K):
    """score_matrix: (n_queries, n_docs) -> list of [(doc_id, score), ...] sorted desc."""
    out = []
    for row in score_matrix:
        idx = np.argpartition(-row, k)[:k]
        idx = idx[np.argsort(-row[idx], kind="stable")]
        out.append([(doc_ids[i], float(row[i])) for i in idx])
    return out


def write_trec(path, claim_ids, ranked, run_name):
    with open(path, "w", encoding="utf-8") as f:
        for cid, docs in zip(claim_ids, ranked):
            assert len(docs) == TOP_K and len({d for d, _ in docs}) == TOP_K, cid
            prev = math.inf
            for r, (did, s) in enumerate(docs, 1):
                s = min(s, prev)          # scores must be non-increasing
                prev = s
                f.write(f"{cid} Q0 {did} {r} {s:.6f} {run_name}\n")


def read_trec(path):
    run = {}
    with open(path, encoding="utf-8") as f:
        for line in f:
            p = line.split()
            run.setdefault(int(p[0]), []).append((int(p[3]), int(p[2]), float(p[4])))
    return {q: [(d, s) for _, d, s in sorted(v)] for q, v in run.items()}


def save_run(name, claim_ids, ranked):
    """Cache a full run (as dict) so later tasks can reuse it."""
    with open(os.path.join(CACHE_DIR, name + ".json"), "w") as f:
        json.dump({str(c): r for c, r in zip(claim_ids, ranked)}, f)


def load_run(name):
    with open(os.path.join(CACHE_DIR, name + ".json")) as f:
        return {int(c): [(int(d), float(s)) for d, s in r] for c, r in json.load(f).items()}


# ---------------------------------------------------------------- metrics
def ndcg_at_k(ranked_ids, rel, k=10):
    dcg = sum(1.0 / math.log2(i + 2) for i, d in enumerate(ranked_ids[:k]) if d in rel)
    idcg = sum(1.0 / math.log2(i + 2) for i in range(min(len(rel), k)))
    return dcg / idcg if idcg else 0.0


def recall_at_k(ranked_ids, rel, k=100):
    return len(set(ranked_ids[:k]) & set(rel)) / len(rel)


def mrr(ranked_ids, rel, k=100):
    for i, d in enumerate(ranked_ids[:k]):
        if d in rel:
            return 1.0 / (i + 1)
    return 0.0


def evaluate(run, qrels):
    """run: {claim_id: [(doc_id, score), ...]}. Averages over claims with gold evidence."""
    n, r, m = [], [], []
    for q, rel in qrels.items():
        ids = [d for d, _ in run[q]]
        n.append(ndcg_at_k(ids, rel))
        r.append(recall_at_k(ids, rel))
        m.append(mrr(ids, rel))
    return {"ndcg10": float(np.mean(n)), "recall100": float(np.mean(r)), "mrr": float(np.mean(m))}


def per_claim_ndcg(run, qrels):
    return {q: ndcg_at_k([d for d, _ in run[q]], rel) for q, rel in qrels.items()}


# ---------------------------------------------------------------- results.json
RESULTS_PATH = os.path.join(ROOT, "results.json")


def update_results(section, values):
    with open(RESULTS_PATH, encoding="utf-8") as f:
        res = json.load(f)
    if isinstance(res.get(section), dict):
        res[section].update(values)
    else:
        res[section] = values
    with open(RESULTS_PATH, "w", encoding="utf-8") as f:
        json.dump(res, f, indent=2)


def log_metrics(name, metrics):
    path = os.path.join(ROOT, "code", "val_metrics.json")
    allm = {}
    if os.path.exists(path):
        with open(path) as f:
            allm = json.load(f)
    allm[name] = metrics
    with open(path, "w") as f:
        json.dump(allm, f, indent=2)
    print(f"[val] {name}: " + ", ".join(f"{k}={v:.4f}" for k, v in metrics.items()))


In [ ]:
%%writefile /content/submission/code/encoders.py
"""Bi-encoder wrappers with a common interface: encode_queries / encode_docs -> L2-normalised float32."""
import numpy as np
import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


class STEncoder:
    """Any sentence-transformers model (optionally with a query instruction prefix)."""

    def __init__(self, name, query_prefix="", max_len=512):
        from sentence_transformers import SentenceTransformer
        self.name = name
        self.model = SentenceTransformer(name, device=DEVICE)
        self.model.max_seq_length = max_len
        if DEVICE == "cuda":
            self.model.half()     # fp16 inference: half the memory, ~2x faster
        self.query_prefix = query_prefix

    def encode_queries(self, qs, bs=64):
        return self.model.encode([self.query_prefix + q for q in qs], batch_size=bs,
                                 normalize_embeddings=True, convert_to_numpy=True).astype(np.float32)

    def encode_docs(self, docs, bs=16):
        texts = [d["title"] + ". " + " ".join(d["abstract"]) for d in docs]
        # sort by length so batches are padded less
        order = np.argsort([-len(t) for t in texts])
        emb = self.model.encode([texts[i] for i in order], batch_size=bs, normalize_embeddings=True,
                                convert_to_numpy=True, show_progress_bar=True).astype(np.float32)
        out = np.empty_like(emb)
        out[order] = emb
        return out

    def n_params(self):
        return sum(p.numel() for p in self.model.parameters())


class MedCPTEncoder:
    """NCBI MedCPT: separate query / article encoders (each ~110M), CLS pooling, inner product."""

    def __init__(self):
        from transformers import AutoModel, AutoTokenizer
        self.name = "ncbi/MedCPT-Query-Encoder + ncbi/MedCPT-Article-Encoder"
        self.qtok = AutoTokenizer.from_pretrained("ncbi/MedCPT-Query-Encoder")
        self.qenc = AutoModel.from_pretrained("ncbi/MedCPT-Query-Encoder").to(DEVICE).eval()
        self.dtok = AutoTokenizer.from_pretrained("ncbi/MedCPT-Article-Encoder")
        self.denc = AutoModel.from_pretrained("ncbi/MedCPT-Article-Encoder").to(DEVICE).eval()
        if DEVICE == "cuda":
            self.qenc.half()
            self.denc.half()

    @torch.no_grad()
    def _run(self, tok, enc, batch, max_len):
        x = tok(*batch, truncation=True, padding=True, return_tensors="pt", max_length=max_len).to(DEVICE)
        return enc(**x).last_hidden_state[:, 0, :].float().cpu().numpy()

    def encode_queries(self, qs, bs=64):
        out = [self._run(self.qtok, self.qenc, (qs[i:i + bs],), 64) for i in range(0, len(qs), bs)]
        return np.concatenate(out).astype(np.float32)

    def encode_docs(self, docs, bs=16):
        from tqdm import tqdm
        order = np.argsort([-len(" ".join(d["abstract"])) for d in docs])
        embs = []
        for i in tqdm(range(0, len(docs), bs), desc="MedCPT docs"):
            b = [docs[j] for j in order[i:i + bs]]
            embs.append(self._run(self.dtok, self.denc,
                                  ([d["title"] for d in b], [" ".join(d["abstract"]) for d in b]), 512))
        emb = np.concatenate(embs).astype(np.float32)
        out = np.empty_like(emb)
        out[order] = emb
        return out

    def n_params(self):
        return sum(p.numel() for m in (self.qenc, self.denc) for p in m.parameters())


BGE_PREFIX = "Represent this sentence for searching relevant passages: "


def get_encoder(kind):
    if kind == "general":
        return STEncoder("BAAI/bge-base-en-v1.5", query_prefix=BGE_PREFIX)
    if kind == "scientific":
        return MedCPTEncoder()
    return STEncoder(kind, query_prefix=BGE_PREFIX)   # path to a fine-tuned bge checkpoint


In [ ]:
%%writefile /content/submission/code/task1_bm25.py
"""Task 1 - corpus exploration + BM25 (own sparse implementation), tuned on validation nDCG@10.

Outputs: predictions/t1_bm25.trec (eval), predictions/val_bm25.trec (val),
         cache/bm25_{val,eval,train}.npy (full score matrices for later tasks)
"""
import itertools
import os
import re
import time
from collections import Counter

import numpy as np
import scipy.sparse as sp

from common import (CACHE_DIR, PRED_DIR, evaluate, load_claims, load_corpus, log_metrics, qrels_of,
                    save_run, set_seed, topk_from_scores, update_results, write_trec)

try:
    import Stemmer
    _stemmer = Stemmer.Stemmer("english")
    stem_words = _stemmer.stemWords
except ImportError:  # fallback
    from nltk.stem import PorterStemmer
    _ps = PorterStemmer()
    stem_words = lambda ws: [_ps.stem(w) for w in ws]  # noqa: E731

STOP = set("""a an and are as at be but by for if in into is it no not of on or such that the their then there
these they this to was will with from we our were which been has have had its than also both can may using
""".split())
TOKEN_RE = re.compile(r"[a-z0-9]+(?:[-.][a-z0-9]+)*")


def tokenize(text, stem=True, stop=True):
    toks = TOKEN_RE.findall(text.lower())
    if stop:
        toks = [t for t in toks if t not in STOP]
    return stem_words(toks) if stem else toks


class BM25:
    def __init__(self, texts, stem=True, stop=True):
        self.stem, self.stop = stem, stop
        toks = [tokenize(t, stem, stop) for t in texts]
        self.vocab = {}
        rows, cols, vals = [], [], []
        for i, ts in enumerate(toks):
            for w, c in Counter(ts).items():
                j = self.vocab.setdefault(w, len(self.vocab))
                rows.append(i); cols.append(j); vals.append(c)
        self.tf = sp.csr_matrix((vals, (rows, cols)), shape=(len(texts), len(self.vocab)), dtype=np.float32)
        self.dl = np.asarray(self.tf.sum(1)).ravel()
        self.avgdl = self.dl.mean()
        df = np.bincount(self.tf.indices, minlength=len(self.vocab))
        n = len(texts)
        self.idf = np.log(1 + (n - df + 0.5) / (df + 0.5)).astype(np.float32)
        self.W = None

    def fit(self, k1=1.2, b=0.75):
        tf = self.tf.tocoo()
        norm = k1 * (1 - b + b * self.dl / self.avgdl)
        w = tf.data * (k1 + 1) / (tf.data + norm[tf.row]) * self.idf[tf.col]
        self.W = sp.csr_matrix((w, (tf.row, tf.col)), shape=self.tf.shape).T.tocsr()  # terms x docs
        return self

    def query_matrix(self, queries):
        rows, cols, vals = [], [], []
        for i, q in enumerate(queries):
            for w, c in Counter(tokenize(q, self.stem, self.stop)).items():
                if w in self.vocab:
                    rows.append(i); cols.append(self.vocab[w]); vals.append(c)
        return sp.csr_matrix((vals, (rows, cols)), shape=(len(queries), len(self.vocab)), dtype=np.float32)

    def score(self, queries):
        return np.asarray((self.query_matrix(queries) @ self.W).todense())


def explore(docs, splits):
    lens = [len(" ".join(d["abstract"]).split()) for d in docs]
    print(f"corpus: {len(docs)} abstracts, words/abstract mean={np.mean(lens):.0f} "
          f"median={np.median(lens):.0f} max={max(lens)}; sentences mean="
          f"{np.mean([len(d['abstract']) for d in docs]):.1f}")
    for name, cl in splits.items():
        if "evidence" not in cl[0]:
            print(f"{name}: {len(cl)} claims (unlabelled)")
            continue
        nei = sum(1 for c in cl if not c["evidence"])
        labs = Counter(e["label"] for c in cl for e in c["evidence"])
        ndocs = Counter(len(c["evidence"]) for c in cl)
        print(f"{name}: {len(cl)} claims, NEI={nei}, labels={dict(labs)}, #gold docs/claim={dict(ndocs)}")


def main():
    set_seed()
    docs, doc_ids, texts = load_corpus()
    splits = {s: load_claims(s) for s in ["train", "val", "eval"]}
    explore(docs, splits)
    val, qrels = splits["val"], qrels_of(splits["val"])
    vq, vids = [c["claim"] for c in val], [int(c["id"]) for c in val]

    best = None
    for stem, stop in [(True, True), (True, False), (False, True)]:
        bm = BM25(texts, stem, stop)
        for k1, b in itertools.product([0.6, 0.9, 1.2, 1.5, 2.0], [0.3, 0.5, 0.75, 0.9, 1.0]):
            S = bm.fit(k1, b).score(vq)
            run = dict(zip(vids, topk_from_scores(S, doc_ids)))
            m = evaluate(run, qrels)
            if best is None or m["ndcg10"] > best[0]["ndcg10"]:
                best = (m, dict(stem=stem, stop=stop, k1=k1, b=b))
    m, cfg = best
    print("best BM25 config:", cfg)
    log_metrics("t1_bm25", m)

    bm = BM25(texts, cfg["stem"], cfg["stop"]).fit(cfg["k1"], cfg["b"])
    t0 = time.time()
    for split in ["val", "eval", "train"]:
        cl = splits[split]
        ids = [int(c["id"]) for c in cl]
        S = bm.score([c["claim"] for c in cl])
        np.save(os.path.join(CACHE_DIR, f"bm25_{split}.npy"), S.astype(np.float32))
        ranked = topk_from_scores(S, doc_ids)
        save_run(f"bm25_{split}", ids, ranked)
        if split == "eval":
            write_trec(os.path.join(PRED_DIR, "t1_bm25.trec"), ids, ranked, "bm25")
        elif split == "val":
            write_trec(os.path.join(PRED_DIR, "val_bm25.trec"), ids, ranked, "bm25")
    print(f"BM25 scoring of all splits took {time.time() - t0:.2f}s")
    settings = (f"own sparse BM25 (Lucene idf), k1={cfg['k1']}, b={cfg['b']}, title+abstract, "
                f"lowercase regex tokens, stopwords={'removed' if cfg['stop'] else 'kept'}, "
                f"{'Snowball stemming' if cfg['stem'] else 'no stemming'}; grid-searched on val nDCG@10")
    update_results("task1", {"bm25_settings": settings, "val_ndcg10": round(m["ndcg10"], 4)})


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/submission/code/task2_dense.py
"""Task 2 - dense retrieval with a general (BGE-base) and a biomedical (MedCPT) bi-encoder.

Outputs: predictions/t2_general.trec, t2_scientific.trec (eval), val_general.trec (val),
         cache/{tag}_{split}.npy score matrices, cache/{tag}_docemb.npy
"""
import os
import sys
import time

import numpy as np
import torch

from common import (CACHE_DIR, PRED_DIR, evaluate, load_claims, load_corpus, log_metrics, qrels_of,
                    save_run, set_seed, topk_from_scores, update_results, write_trec)
from encoders import DEVICE, get_encoder


def run_dense(kind, tag, trec_eval=None, trec_val=None):
    """Encode corpus + all claim splits with encoder `kind`; returns (val metrics, cost stats)."""
    set_seed()
    docs, doc_ids, _ = load_corpus()
    enc = get_encoder(kind)

    if DEVICE == "cuda":
        torch.cuda.synchronize()
    t0 = time.time()
    D = enc.encode_docs(docs)
    if DEVICE == "cuda":
        torch.cuda.synchronize()
    index_time = time.time() - t0
    np.save(os.path.join(CACHE_DIR, f"{tag}_docemb.npy"), D)

    metrics = None
    for split in ["val", "eval", "train"]:
        cl = load_claims(split)
        ids = [int(c["id"]) for c in cl]
        S = enc.encode_queries([c["claim"] for c in cl]) @ D.T
        np.save(os.path.join(CACHE_DIR, f"{tag}_{split}.npy"), S.astype(np.float32))
        ranked = topk_from_scores(S, doc_ids)
        save_run(f"{tag}_{split}", ids, ranked)
        if split == "val":
            metrics = evaluate(dict(zip(ids, ranked)), qrels_of(cl))
            log_metrics(tag, metrics)
            if trec_val:
                write_trec(os.path.join(PRED_DIR, trec_val), ids, ranked, tag)
        if split == "eval" and trec_eval:
            write_trec(os.path.join(PRED_DIR, trec_eval), ids, ranked, tag)

    # online query latency: one claim at a time, encode + exact search + top-100
    qs = [c["claim"] for c in load_claims("val")]
    enc.encode_queries(qs[:5], bs=1)  # warm-up
    t0 = time.time()
    for q in qs:
        s = enc.encode_queries([q], bs=1) @ D.T
        topk_from_scores(s, doc_ids)
    latency_ms = (time.time() - t0) / len(qs) * 1000
    wbytes = 2 if DEVICE == "cuda" else 4
    memory_mb = (enc.n_params() * wbytes + D.nbytes) / 2 ** 20   # model weights (fp16 on GPU) + flat fp32 index
    cost = {"index_time_sec": round(index_time, 2), "query_latency_ms": round(latency_ms, 2),
            "memory_mb": round(memory_mb, 1), "index_mb": round(D.nbytes / 2 ** 20, 1)}
    print(f"{tag}: {cost}")
    del enc
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    return metrics, cost


def main():
    names = {"general": "BAAI/bge-base-en-v1.5 (109M, query instruction prefix, max 512 tokens)",
             "scientific": "ncbi/MedCPT-Query-Encoder + ncbi/MedCPT-Article-Encoder (110M each, "
                           "PubMed-trained, CLS + inner product)"}
    out = {"general_model": names["general"], "scientific_model": names["scientific"],
           "index_time_sec": {}, "query_latency_ms": {}, "memory_mb": {}}
    kinds = sys.argv[1:] or ["general", "scientific"]
    for kind in kinds:
        m, cost = run_dense(kind, kind, trec_eval=f"t2_{kind}.trec",
                            trec_val="val_general.trec" if kind == "general" else None)
        out[f"val_ndcg10_{kind}"] = round(m["ndcg10"], 4)
        for k in ["index_time_sec", "query_latency_ms", "memory_mb"]:
            out[k][kind] = cost[k]
    import json
    from common import RESULTS_PATH
    with open(RESULTS_PATH) as f:
        prev = json.load(f)["task2"]
    for k in ["index_time_sec", "query_latency_ms", "memory_mb"]:
        prev[k].update(out.pop(k))
    prev.update(out)
    update_results("task2", prev)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/submission/code/task3_hybrid_rerank.py
"""Task 3 - hybrid (BM25 + dense) fusion and cross-encoder reranking.

Fusion weights are grid-searched on the TRAIN claims (all inputs are pretrained models, so train is
clean held-out data for them); the fusion method and reranker depth k are then chosen on val nDCG@10.
Outputs: predictions/t3_hybrid.trec, t3_rerank.trec (eval); cache/hybrid_*, rerank_* runs.
"""
import itertools
import json
import os
import time

import numpy as np

from common import (CACHE_DIR, PRED_DIR, evaluate, load_claims, load_corpus, load_run, log_metrics,
                    qrels_of, save_run, set_seed, topk_from_scores, update_results, write_trec)

SOURCES = ["bm25", "general", "scientific"]
RERANKERS = ["BAAI/bge-reranker-base", "cross-encoder/ms-marco-MiniLM-L-12-v2", "ncbi/MedCPT-Cross-Encoder"]
KS = [5, 10, 20, 30, 50, 100]


def load_scores(split):
    return {s: np.load(os.path.join(CACHE_DIR, f"{s}_{split}.npy")) for s in SOURCES}


def zscore(S):
    return (S - S.mean(1, keepdims=True)) / (S.std(1, keepdims=True) + 1e-9)


def minmax(S):
    lo, hi = S.min(1, keepdims=True), S.max(1, keepdims=True)
    return (S - lo) / (hi - lo + 1e-9)


def rrf(S, k=60):
    ranks = np.argsort(np.argsort(-S, axis=1), axis=1) + 1
    return 1.0 / (k + ranks)


NORMS = {"zscore": zscore, "minmax": minmax, "rrf": rrf}


def fuse(scores, norm, w):
    return sum(wi * NORMS[norm](scores[s]) for s, wi in zip(SOURCES, w) if wi > 0)


def ids_qrels(split):
    cl = load_claims(split)
    return [int(c["id"]) for c in cl], qrels_of(cl), cl


def search_weights(norm, doc_ids, tr_scores, tr_ids, tr_q):
    # BM25 must take part (the task asks for a BM25 + dense hybrid), plus at least one dense model
    grid = [tuple(round(float(x), 1) for x in w) for w in itertools.product(np.arange(0, 1.01, 0.1), repeat=3)
            if abs(sum(w) - 1) < 1e-6 and w[0] > 0 and (w[1] > 0 or w[2] > 0)]
    best = None
    for w in grid:
        m = evaluate(dict(zip(tr_ids, topk_from_scores(fuse(tr_scores, norm, w), doc_ids))), tr_q)
        if best is None or m["ndcg10"] > best[0]:
            best = (m["ndcg10"], w)
    return best


def main():
    set_seed()
    docs, doc_ids, _ = load_corpus()
    doc_text = {int(d["doc_id"]): d["title"] + ". " + " ".join(d["abstract"]) for d in docs}
    tr_ids, tr_q, _ = ids_qrels("train")
    va_ids, va_q, va_cl = ids_qrels("val")
    ev_ids, _, ev_cl = ids_qrels("eval")
    tr_s, va_s, ev_s = load_scores("train"), load_scores("val"), load_scores("eval")

    # ---------------- hybrid
    candidates = []
    for norm in NORMS:
        tr_ndcg, w = search_weights(norm, doc_ids, tr_s, tr_ids, tr_q)
        m = evaluate(dict(zip(va_ids, topk_from_scores(fuse(va_s, norm, w), doc_ids))), va_q)
        print(f"hybrid {norm} w(bm25,gen,sci)={w}: train nDCG={tr_ndcg:.4f}  val={m}")
        candidates.append((m["ndcg10"], norm, w, m))
    _, norm, w, m = max(candidates, key=lambda x: x[0])
    log_metrics("t3_hybrid", m)
    if norm == "rrf":
        method = (f"weighted reciprocal rank fusion over the full corpus: score = {w[0]}/(60+rank_BM25) + "
                  f"{w[1]}/(60+rank_BGE) + {w[2]}/(60+rank_MedCPT); weights grid-searched on train claims, "
                  f"RRF chosen over z-score / min-max score interpolation on val")
    else:
        method = (f"weighted sum of per-query {norm}-normalised scores over the full corpus: "
                  f"{w[0]}*BM25 + {w[1]}*BGE-base + {w[2]}*MedCPT (weights grid-searched on train claims, "
                  f"normalisation chosen on val)")
    t0 = time.time()
    for split, S, ids in [("val", va_s, va_ids), ("eval", ev_s, ev_ids), ("train", tr_s, tr_ids)]:
        ranked = topk_from_scores(fuse(S, norm, w), doc_ids)
        save_run(f"hybrid_{split}", ids, ranked)
        if split == "eval":
            write_trec(os.path.join(PRED_DIR, "t3_hybrid.trec"), ids, ranked, "hybrid")
    fusion_ms = (time.time() - t0) / (len(va_ids) + len(ev_ids) + len(tr_ids)) * 1000

    # ---------------- rerank the best first-stage system (by val nDCG@10)
    with open(os.path.join(os.path.dirname(__file__), "val_metrics.json")) as f:
        vm = json.load(f)
    first = max(["t1_bm25", "general", "scientific", "t3_hybrid"], key=lambda k: vm[k]["ndcg10"])
    first_run = {"t1_bm25": "bm25", "t3_hybrid": "hybrid"}.get(first, first)
    print("first stage for reranking:", first_run)

    best = None
    ce_scores = {}
    for name in RERANKERS:
        base = load_run(f"{first_run}_val")
        cache = os.path.join(CACHE_DIR, f"ce_{name.replace('/', '__')}_{first_run}_val.json")
        if os.path.exists(cache):
            with open(cache) as f:
                sc = {int(q): v for q, v in json.load(f).items()}
        else:
            ce = load_ce(name)
            claim = {int(c["id"]): c["claim"] for c in va_cl}
            sc = {q: ce.predict([(claim[q], doc_text[d]) for d, _ in base[q]], batch_size=32).tolist()
                  for q in va_ids}
            with open(cache, "w") as f:
                json.dump(sc, f)
            del ce
        ce_scores[name] = sc
        print(f"scored val top-100 with {name}", flush=True)
        for k in KS:
            m = evaluate(rerank(base, sc, k), va_q)
            print(f"rerank {name} k={k}: {m}")
            if best is None or m["ndcg10"] > best[0]["ndcg10"]:
                best = (m, name, k)
    m, name, k = best
    log_metrics("t3_rerank", m)

    # latency of the reranking stage alone (k pairs per claim, batch = k)
    ce = load_ce(name)
    base_val = load_run(f"{first_run}_val")
    claim = {int(c["id"]): c["claim"] for c in va_cl}
    ce.predict([(claim[va_ids[0]], doc_text[d]) for d, _ in base_val[va_ids[0]][:k]])  # warm-up
    t0 = time.time()
    for q in va_ids[:60]:
        ce.predict([(claim[q], doc_text[d]) for d, _ in base_val[q][:k]], batch_size=k)
    rerank_ms = (time.time() - t0) / 60 * 1000

    for split, cl in [("val", va_cl), ("eval", ev_cl)]:
        base = load_run(f"{first_run}_{split}")
        if split == "val" and name in ce_scores:
            sc = ce_scores[name]
        else:
            claim = {int(c["id"]): c["claim"] for c in cl}
            sc = {q: ce.predict([(claim[q], doc_text[d]) for d, _ in base[q][:k]], batch_size=32)
                  for q in base}
        rr = rerank(base, sc, k)
        ids = [int(c["id"]) for c in cl]
        save_run(f"rerank_{split}", ids, [rr[q] for q in ids])
        if split == "eval":
            write_trec(os.path.join(PRED_DIR, "t3_rerank.trec"), ids, [rr[q] for q in ids], "rerank")

    update_results("task3", {
        "hybrid_method": method,
        "reranker_model": f"{name} (cross-encoder over claim + title/abstract, max 512 tokens), "
                          f"first stage = {first_run}",
        "rerank_top_k": k,
        "val_ndcg10_hybrid": round(vm["t3_hybrid"]["ndcg10"] if "t3_hybrid" in vm else 0, 4),
        "val_ndcg10_rerank": round(m["ndcg10"], 4),
        "rerank_query_latency_ms": round(rerank_ms, 1),
    })
    with open(os.path.join(CACHE_DIR, "task3_meta.json"), "w") as f:
        json.dump({"first_stage": first_run, "fusion_ms_per_query": fusion_ms, "reranker": name, "k": k,
                   "rerank_ms": rerank_ms}, f)


def load_ce(name):
    from sentence_transformers import CrossEncoder
    ce = CrossEncoder(name, max_length=512)
    if ce.model.device.type == "cuda":
        ce.model.half()
    return ce


def rerank(base, sc, k):
    """Top-k of base in cross-encoder order with scores shifted above the tail; tail keeps base order."""
    out = {}
    for q, docs in base.items():
        s = np.asarray(sc[q][:k], dtype=float)
        order = np.argsort(-s, kind="stable")
        tail = docs[k:]
        # map so that every head score > every tail score, keeping monotonic non-increasing scores
        tail_top = tail[0][1] if tail else 0.0
        head = [(docs[i][0], float(s[i] - s.min() + 1 + tail_top)) for i in order]
        out[q] = head + list(tail)
    return out


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/submission/code/task4_finetune.py
"""Task 4 - fine-tune BGE-base on the training claims, once with random and once with hard negatives.

Loss: MultipleNegativesRankingLoss (in-batch negatives + one explicit negative per pair).
  random : explicit negative = random corpus abstract that is not gold for the claim
  hard   : explicit negative = sampled from the top-30 of BM25 / BGE / MedCPT on the claim (non-gold)
After every epoch we evaluate val nDCG@10 (and train nDCG@10) to detect overfitting and keep the best
epoch's checkpoint.
Outputs: predictions/t4_random.trec, t4_hard.trec (eval); cache/ft_{random,hard}_* runs.
"""
import json
import os
import random
import sys

import numpy as np
import torch

from common import (CACHE_DIR, PRED_DIR, SEED, evaluate, load_claims, load_corpus, load_run, log_metrics,
                    qrels_of, set_seed, topk_from_scores, update_results)
from encoders import BGE_PREFIX, DEVICE

BASE = "BAAI/bge-base-en-v1.5"
EPOCHS = 2
BATCH = 16
LR = 1e-5   # 2e-5 overfit after one epoch (val nDCG@10 0.891 -> 0.876 -> 0.865 while train rose)
MAX_LEN = 256


def doc_str(d):
    return d["title"] + ". " + " ".join(d["abstract"])


def build_examples(mode, docs, train):
    rng = random.Random(SEED)
    by_id = {int(d["doc_id"]): d for d in docs}
    all_ids = list(by_id)
    mined = {}
    if mode == "hard":
        runs = [load_run(f"{s}_train") for s in ["bm25", "general", "scientific"]]
        for c in train:
            q = int(c["id"])
            gold = {int(e["doc_id"]) for e in c["evidence"]}
            cands = []
            for r in runs:
                cands += [d for d, _ in r[q][:30] if d not in gold and d not in cands]
            mined[q] = cands
    ex = []
    for c in train:
        q = int(c["id"])
        gold = {int(e["doc_id"]) for e in c["evidence"]}
        for g in gold:
            if mode == "hard":
                neg = rng.choice(mined[q][:20])   # hard but not only the very top (false-negative risk)
            else:
                neg = rng.choice(all_ids)
                while neg in gold:
                    neg = rng.choice(all_ids)
            ex.append((BGE_PREFIX + c["claim"], doc_str(by_id[g]), doc_str(by_id[neg])))
    return ex


def evaluate_model(model, docs, doc_ids, splits=("val", "train")):
    """One corpus encoding (fp16 autocast, small batches to stay inside GPU memory), metrics per split."""
    model.eval()
    with torch.autocast("cuda", dtype=torch.float16, enabled=DEVICE == "cuda"):
        D = model.encode([doc_str(d) for d in docs], batch_size=16, normalize_embeddings=True,
                         convert_to_numpy=True)
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    out = {}
    for split in splits:
        cl = load_claims(split)
        with torch.autocast("cuda", dtype=torch.float16, enabled=DEVICE == "cuda"):
            Q = model.encode([BGE_PREFIX + c["claim"] for c in cl], batch_size=64, normalize_embeddings=True,
                             convert_to_numpy=True)
        D, Q = D.astype("float32"), Q.astype("float32")
        ids = [int(c["id"]) for c in cl]
        out[split] = evaluate(dict(zip(ids, topk_from_scores(Q @ D.T, doc_ids))), qrels_of(cl))
    return out


def train(mode):
    from sentence_transformers import InputExample, SentenceTransformer, losses
    from torch.utils.data import DataLoader

    set_seed()
    docs, doc_ids, _ = load_corpus()
    train_cl = [c for c in load_claims("train") if c["evidence"]]
    ex = build_examples(mode, docs, train_cl)
    print(f"{mode}: {len(ex)} training triples from {len(train_cl)} claims")

    model = SentenceTransformer(BASE, device=DEVICE)
    model.max_seq_length = MAX_LEN
    loader = DataLoader([InputExample(texts=list(t)) for t in ex], shuffle=True, batch_size=BATCH,
                        generator=torch.Generator().manual_seed(SEED))
    loss = losses.MultipleNegativesRankingLoss(model)

    out_dir = os.path.join(CACHE_DIR, f"ft_{mode}")
    model.max_seq_length = 512
    e0 = evaluate_model(model, docs, doc_ids)
    history = [{"epoch": 0, **e0}]
    best = (-1.0, 0)    # best *fine-tuned* epoch; epoch 0 (base model) is logged for reference only
    print(f"epoch 0: train nDCG={e0['train']['ndcg10']:.4f}  val {e0['val']}")
    for ep in range(1, EPOCHS + 1):
        model.max_seq_length = MAX_LEN
        model.fit(train_objectives=[(loader, loss)], epochs=1, warmup_steps=int(0.1 * len(loader)) if ep == 1
                  else 0, optimizer_params={"lr": LR}, show_progress_bar=True, use_amp=DEVICE == "cuda")
        model.max_seq_length = 512
        e = evaluate_model(model, docs, doc_ids)
        mv, mt = e["val"], e["train"]
        history.append({"epoch": ep, **e})
        print(f"epoch {ep}: train nDCG={mt['ndcg10']:.4f}  val {mv}")
        if mv["ndcg10"] > best[0]:
            best = (mv["ndcg10"], ep)
            model.save(out_dir)
    with open(os.path.join(CACHE_DIR, f"ft_{mode}_history.json"), "w") as f:
        json.dump({"history": history, "best_epoch": best[1]}, f, indent=2)
    return out_dir, history, best


def main():
    from task2_dense import run_dense
    modes = sys.argv[1:] or ["random", "hard"]
    summary = {}
    for mode in modes:
        out_dir, history, best = train(mode)
        m, cost = run_dense(out_dir, f"ft_{mode}", trec_eval=f"t4_{mode}.trec")
        log_metrics(f"t4_{mode}", m)
        summary[f"val_ndcg10_{mode}_negatives"] = round(m["ndcg10"], 4)
        summary[f"{mode}_best_epoch"] = best[1]
        if DEVICE == "cuda":
            torch.cuda.empty_cache()
    summary["base_model"] = (f"{BASE} fine-tuned with MultipleNegativesRankingLoss (in-batch + 1 explicit "
                             f"negative), lr={LR}, batch={BATCH}, max_len={MAX_LEN}, up to {EPOCHS} epochs, "
                             f"best epoch by val nDCG@10")
    update_results("task4", summary)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/submission/code/task5_verify.py
"""Task 5 - claim verification over the best retriever's top abstracts + error analysis.

Pipeline per (claim, retrieved abstract):
  1. zero-shot NLI (DeBERTa-v3-base MNLI/FEVER/ANLI) on every abstract sentence (premise) vs the claim
     (hypothesis), plus once on the whole abstract;
  2. features = max/top-2 entailment & contradiction over sentences, abstract-level probs, retrieval rank;
  3. a 3-way logistic regression (SUPPORT / CONTRADICT / NONE) trained on the TRAIN claims, whose
     decision margin and number of abstracts considered are tuned with 5-fold out-of-fold predictions
     on train. Validation is only used to report the final number and for the error analysis.
Outputs: predictions/t5_verification.csv, t5_val_verification.csv, t5_val_errors.csv
"""
import csv
import json
import os
import re

import numpy as np
import torch
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold

from common import CACHE_DIR, PRED_DIR, ROOT, load_claims, load_corpus, load_run, set_seed, update_results
from encoders import DEVICE

NLI = "MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli"
CAND = 3          # abstracts per claim passed to the verifier (max rows allowed per claim)
LABELS = ["NONE", "SUPPORT", "CONTRADICT"]
NEG_RE = re.compile(r"\b(not|no|never|none|without|fail|fails|failed|cannot|lack|lacks|absence|"
                    r"neither|nor|unable|independent|unaffected|n't)\b", re.I)
NUM_RE = re.compile(r"\d|percent|%|fold|times|half|double|twice|one|two|three|ten|hundred|thousand", re.I)


class NLIScorer:
    def __init__(self):
        from transformers import AutoModelForSequenceClassification, AutoTokenizer
        self.tok = AutoTokenizer.from_pretrained(NLI)
        self.model = AutoModelForSequenceClassification.from_pretrained(NLI).to(DEVICE).eval()
        l2i = {k.lower(): v for k, v in self.model.config.label2id.items()}
        self.idx = [l2i["entailment"], l2i["neutral"], l2i["contradiction"]]
        cache = os.path.join(CACHE_DIR, "nli_cache.json")
        self.cache_path = cache
        self.cache = json.load(open(cache)) if os.path.exists(cache) else {}

    @torch.no_grad()
    def probs(self, premises, hyp, bs=32):
        out = []
        for i in range(0, len(premises), bs):
            x = self.tok(premises[i:i + bs], [hyp] * len(premises[i:i + bs]), truncation="only_first",
                         max_length=512, padding=True, return_tensors="pt").to(DEVICE)
            p = torch.softmax(self.model(**x).logits.float(), -1)[:, self.idx]
            out.append(p.cpu().numpy())
        return np.concatenate(out)          # columns: entail, neutral, contra

    def doc(self, claim, d):
        key = f"{claim}|||{d['doc_id']}"
        if key not in self.cache:
            sent = self.probs(d["abstract"], claim)
            whole = self.probs([d["title"] + ". " + " ".join(d["abstract"])], claim)[0]
            self.cache[key] = {"sent": sent.tolist(), "whole": whole.tolist()}
        return np.asarray(self.cache[key]["sent"]), np.asarray(self.cache[key]["whole"])

    def save(self):
        with open(self.cache_path, "w") as f:
            json.dump(self.cache, f)


def features(sent, whole, rank):
    e, c = sent[:, 0], sent[:, 2]
    es, cs = np.sort(e)[::-1], np.sort(c)[::-1]
    best = sent[np.argmax(e + c)]
    return [es[0], es[:2].mean(), cs[0], cs[:2].mean(), best[0], best[2], es[0] - cs[0],
            whole[0], whole[1], whole[2], 1.0 / rank, float(rank == 1)]


def build(split, run, nli, by_id, train=False):
    """Rows of (claim_id, doc_id, rank, features, label_idx)."""
    rows = []
    for c in load_claims(split):
        q = int(c["id"])
        gold = {int(e["doc_id"]): e["label"] for e in c.get("evidence") or []}
        docs = [d for d, _ in run[q][:CAND]]
        if train:          # also learn from gold abstracts the retriever missed
            docs += [g for g in gold if g not in docs]
        for r, d in enumerate(docs, 1):
            sent, whole = nli.doc(c["claim"], by_id[d])
            y = LABELS.index(gold[d]) if d in gold else 0
            rows.append((q, d, min(r, CAND + 1), features(sent, whole, min(r, CAND + 1)), y))
    return rows


def decide(rows, proba, margin, n_docs):
    """Pick label per (claim, doc); emit SUPPORT/CONTRADICT only if it beats NONE by `margin`."""
    preds = []
    for (q, d, r, _, _), p in zip(rows, proba):
        if r > n_docs:
            continue
        lab = 1 + int(np.argmax(p[1:]))
        if p[lab] - p[0] > margin:
            preds.append((q, d, LABELS[lab], p[lab]))
    out = {}
    for q, d, lab, s in sorted(preds, key=lambda x: -x[3]):
        if len(out.setdefault(q, [])) < 3:
            out[q].append((d, lab))
    return {(q, d, lab) for q, v in out.items() for d, lab in v}


def gold_triples(claims):
    return {(int(c["id"]), int(e["doc_id"]), e["label"]) for c in claims for e in c.get("evidence") or []}


def prf(pred, gold):
    tp = len(pred & gold)
    p = tp / len(pred) if pred else 0.0
    r = tp / len(gold) if gold else 0.0
    return p, r, (2 * p * r / (p + r) if p + r else 0.0)


def categorize(c, pred_pairs, retrieved, by_id):
    gold = {int(e["doc_id"]): e["label"] for e in c["evidence"]}
    claim = c["claim"]
    if gold and any(g not in retrieved for g in gold):
        return "retrieval_miss"
    correct = {d for d, l in pred_pairs.items() if gold.get(d) == l}
    if len(gold) > 1 and correct and len(correct) < len(gold):
        return "needs_multiple_docs"
    flipped = [d for d, l in pred_pairs.items() if d in gold and gold[d] != l]
    missed = [d for d in gold if d not in pred_pairs]
    false_pos = [d for d in pred_pairs if d not in gold]
    if flipped or missed:
        d = (flipped or missed)[0]
        rat = " ".join(by_id[d]["abstract"][i] for e in c["evidence"] if int(e["doc_id"]) == d
                       for i in e.get("sentences", []) if i < len(by_id[d]["abstract"]))
        if NEG_RE.search(claim) or (flipped and NEG_RE.search(rat)):
            return "negation"
        if NUM_RE.search(claim):
            return "numerical"
        return "other"
    if false_pos:
        return "numerical" if re.search(r"\d", claim) else "entity_mismatch"
    return "other"


def main():
    set_seed()
    docs, _, _ = load_corpus()
    by_id = {int(d["doc_id"]): d for d in docs}
    with open(os.path.join(ROOT, "code", "val_metrics.json")) as f:
        vm = json.load(f)
    names = {"t1_bm25": "bm25", "general": "general", "scientific": "scientific", "t3_hybrid": "hybrid",
             "t3_rerank": "rerank", "t4_random": "ft_random", "t4_hard": "ft_hard"}
    best_key = max((k for k in names if k in vm), key=lambda k: vm[k]["ndcg10"])
    retr = names[best_key]
    print(f"best retriever by val nDCG@10: {best_key} ({vm[best_key]['ndcg10']:.4f})")

    nli = NLIScorer()
    # train candidates come from the (pretrained-only) hybrid run: fine-tuned retrievers saw these claims
    train_rows = build("train", load_run("hybrid_train"), nli, by_id, train=True)
    val_rows = build("val", load_run(f"{retr}_val"), nli, by_id)
    eval_rows = build("eval", load_run(f"{retr}_eval"), nli, by_id)
    nli.save()

    X = np.array([r[3] for r in train_rows]); y = np.array([r[4] for r in train_rows])
    groups = np.array([r[0] for r in train_rows])
    clf_args = dict(C=1.0, max_iter=2000, class_weight="balanced")
    oof = np.zeros((len(y), 3))
    for tr, te in GroupKFold(5).split(X, y, groups):
        oof[te] = LogisticRegression(**clf_args).fit(X[tr], y[tr]).predict_proba(X[te])
    train_cl = load_claims("train")
    g_train = gold_triples(train_cl)
    best = None
    for margin in np.arange(-0.3, 0.61, 0.05):
        for n_docs in [1, 2, 3]:
            # only evaluate on rows a retriever would actually produce (rank <= CAND)
            f1 = prf(decide(train_rows, oof, margin, n_docs), g_train)[2]
            if best is None or f1 > best[0]:
                best = (f1, round(float(margin), 2), n_docs)
    print(f"train OOF F1={best[0]:.4f} margin={best[1]} n_docs={best[2]}")
    _, margin, n_docs = best
    clf = LogisticRegression(**clf_args).fit(X, y)

    val_cl = load_claims("val")
    vp = decide(val_rows, clf.predict_proba(np.array([r[3] for r in val_rows])), margin, n_docs)
    p, r, f1 = prf(vp, gold_triples(val_cl))
    print(f"val verification P={p:.4f} R={r:.4f} F1={f1:.4f}")
    ep = decide(eval_rows, clf.predict_proba(np.array([r[3] for r in eval_rows])), margin, n_docs)

    for name, triples in [("t5_verification.csv", ep), ("t5_val_verification.csv", vp)]:
        with open(os.path.join(PRED_DIR, name), "w", newline="", encoding="utf-8") as f:
            w = csv.writer(f)
            w.writerow(["claim_id", "doc_id", "label"])
            w.writerows(sorted(triples))

    # ---------------- error analysis on validation claims
    val_run = load_run(f"{retr}_val")
    errors = []
    for c in val_cl:
        q = int(c["id"])
        pred_pairs = {d: l for qq, d, l in vp if qq == q}
        gold_pairs = {int(e["doc_id"]): e["label"] for e in c["evidence"]}
        if pred_pairs != gold_pairs:
            retrieved = [d for d, _ in val_run[q][:n_docs]]
            errors.append((q, categorize(c, pred_pairs, retrieved, by_id)))
    with open(os.path.join(PRED_DIR, "t5_val_errors.csv"), "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["claim_id", "category"])
        w.writerows(errors)
    from collections import Counter
    cnt = Counter(c for _, c in errors)
    print(f"{len(errors)} / {len(val_cl)} val claims wrong; categories: {dict(cnt)}")
    with open(os.path.join(CACHE_DIR, "task5_meta.json"), "w") as f:
        json.dump({"retriever": best_key, "margin": margin, "n_docs": n_docs, "train_oof_f1": best[0],
                   "val_p": p, "val_r": r, "val_f1": f1, "n_errors": len(errors),
                   "categories": dict(cnt), "n_val_pred": len(vp), "n_eval_pred": len(ep)}, f, indent=2)

    update_results("task5", {
        "retriever_used": f"{best_key} (best val nDCG@10 = {vm[best_key]['ndcg10']:.4f}); top-{n_docs} abstracts",
        "nli_model": f"{NLI} (184M), sentence-level + whole-abstract NLI, aggregated by a 3-way logistic "
                     f"regression trained on train claims",
        "zero_shot_or_finetuned": "zero-shot NLI model; decision layer (logistic regression + margin) "
                                  "trained/tuned on the train claims only",
        "val_f1": round(f1, 4),
    })


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/submission/code/report_analysis.py
"""Q1 per-claim comparison (val_bm25.trec vs val_general.trec) + validation ablation table."""
import json
import os

from common import PRED_DIR, ROOT, load_claims, load_corpus, per_claim_ndcg, qrels_of, read_trec, update_results


def main():
    val = load_claims("val")
    qrels = qrels_of(val)
    claim = {int(c["id"]): c["claim"] for c in val}
    docs, _, _ = load_corpus()
    title = {int(d["doc_id"]): d["title"] for d in docs}
    bm = read_trec(os.path.join(PRED_DIR, "val_bm25.trec"))
    ge = read_trec(os.path.join(PRED_DIR, "val_general.trec"))
    nb, ng = per_claim_ndcg(bm, qrels), per_claim_ndcg(ge, qrels)
    diff = sorted(qrels, key=lambda q: nb[q] - ng[q])
    bm_wins, ge_wins = diff[::-1][:2], diff[:2]
    print(f"BM25 better on {sum(nb[q] > ng[q] for q in qrels)} claims, BGE better on "
          f"{sum(ng[q] > nb[q] for q in qrels)}, ties {sum(nb[q] == ng[q] for q in qrels)}")
    for tag, qs in [("BM25 wins", bm_wins), ("general wins", ge_wins)]:
        for q in qs:
            gold = list(qrels[q])
            rb = [d for d, _ in bm[q]].index(gold[0]) + 1 if gold[0] in [d for d, _ in bm[q]] else ">100"
            rg = [d for d, _ in ge[q]].index(gold[0]) + 1 if gold[0] in [d for d, _ in ge[q]] else ">100"
            print(f"\n[{tag}] claim {q}: nDCG bm25={nb[q]:.3f} general={ng[q]:.3f} "
                  f"(gold rank bm25={rb}, general={rg})\n  claim: {claim[q]}\n  gold:  {title[gold[0]]}"
                  f"\n  bm25 top1:    {title[bm[q][0][0]]}\n  general top1: {title[ge[q][0][0]]}")
    update_results("report_q1", {"bm25_wins": bm_wins, "general_wins": ge_wins})

    with open(os.path.join(ROOT, "code", "val_metrics.json")) as f:
        vm = json.load(f)
    print("\nValidation ablation")
    print(f"{'run':14s} {'nDCG@10':>8s} {'R@100':>8s} {'MRR':>8s}")
    for k, m in vm.items():
        print(f"{k:14s} {m['ndcg10']:8.4f} {m['recall100']:8.4f} {m['mrr']:8.4f}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/submission/code/make_report.py
"""Write ../report.md (Q1-Q5, <= 100 words each) from results.json, val_metrics.json and cached metadata.

Run after task5_verify.py and report_analysis.py. The text is filled from the measured numbers.
"""
import json
import os

from common import CACHE_DIR, PRED_DIR, ROOT, load_claims, per_claim_ndcg, qrels_of, read_trec


def jload(p, default=None):
    return json.load(open(p)) if os.path.exists(p) else default


def main():
    res = jload(os.path.join(ROOT, "results.json"))
    vm = jload(os.path.join(ROOT, "code", "val_metrics.json"))
    t3 = jload(os.path.join(CACHE_DIR, "task3_meta.json"), {})
    t5 = jload(os.path.join(CACHE_DIR, "task5_meta.json"), {})
    hist = {m: jload(os.path.join(CACHE_DIR, f"ft_{m}_history.json"), {}) for m in ["random", "hard"]}
    val = load_claims("val")
    qrels = qrels_of(val)
    claim = {int(c["id"]): c["claim"] for c in val}
    nb = per_claim_ndcg(read_trec(os.path.join(PRED_DIR, "val_bm25.trec")), qrels)
    ng = per_claim_ndcg(read_trec(os.path.join(PRED_DIR, "val_general.trec")), qrels)
    q1 = res["report_q1"]

    def short(q, n=9):
        w = claim[q].split()
        return " ".join(w[:n]) + ("…" if len(w) > n else "")

    a, b = q1["bm25_wins"]
    c, d = q1["general_wins"]
    Q1 = (f"BM25 wins: {a} ({nb[a]:.2f} vs {ng[a]:.2f}, \"{short(a, 6)}\") and {b} ({nb[b]:.2f} vs {ng[b]:.2f}). "
          f"General (BGE) wins: {c} ({ng[c]:.2f} vs {nb[c]:.2f}, \"{short(c, 6)}\") and {d} "
          f"({ng[d]:.2f} vs {nb[d]:.2f}). BM25 wins when the claim shares rare exact tokens (gene/drug names, "
          f"acronyms) with the gold abstract, which the dense model blurs into topically similar papers. BGE wins "
          f"when the claim paraphrases the abstract, where lexical overlap is low.")

    h, r = vm["t3_hybrid"]["ndcg10"], vm["t3_rerank"]["ndcg10"]
    best_single = max(vm["general"]["ndcg10"], vm["scientific"]["ndcg10"], vm["t1_bm25"]["ndcg10"])
    gain_h, gain_r = h - best_single, r - h
    big = "reranking" if gain_r > gain_h else "hybrid fusion"
    lat = res["task3"]["rerank_query_latency_ms"]
    dense_lat = max(res["task2"]["query_latency_ms"].values())
    Q2 = (f"Hybrid fusion moved val nDCG@10 from {best_single:.3f} (best single retriever) to {h:.3f} "
          f"(+{gain_h:.3f}); cross-encoder reranking of the top-{res['task3']['rerank_top_k']} moved it to "
          f"{r:.3f} ({gain_r:+.3f}). The largest gain came from {big}. Fusion is almost free (one extra dense "
          f"query, ~{dense_lat:.0f} ms, plus a vector sum), while reranking adds ~{lat:.0f} ms per claim on our GPU "
          f"because every (claim, abstract) pair needs a full 512-token cross-encoder pass. "
          f"General-domain rerankers (bge-reranker-base, MiniLM) actually hurt; only the biomedical MedCPT "
          f"cross-encoder helped.")

    vr, vh = vm["t4_random"]["ndcg10"], vm["t4_hard"]["ndcg10"]
    def curve(m):
        return ", ".join(f"ep{x['epoch']} {x['val']['ndcg10']:.3f}" + (f"/{x['train']['ndcg10']:.3f}" if 'train' in x else "")
                         for x in hist[m].get("history", []))
    Q3 = (f"Hard negatives {'helped' if vh > vr else 'did not help'}: val nDCG@10 {vh:.3f} vs {vr:.3f} with random "
          f"negatives (base model {vm['general']['ndcg10']:.3f}). We logged val/train nDCG@10 after every epoch "
          f"(random: {curve('random')}; hard: {curve('hard')}). Overfitting shows as train nDCG rising while val "
          f"stalls or drops; we kept the best-val epoch. With only ~440 claim–abstract pairs the gap opens "
          f"quickly, and some mined 'hard negatives' are unlabelled positives, which limits the benefit.")

    cats = t5.get("categories", {})
    n_err = t5.get("n_errors", 0)
    rm = cats.get("retrieval_miss", 0)
    Q4 = (f"{rm} of {n_err} validation verification errors are retrieval_miss "
          f"({(rm / n_err if n_err else 0):.0%}). Full breakdown: "
          + ", ".join(f"{k} {v}" for k, v in sorted(cats.items(), key=lambda x: -x[1]))
          + f". Validation F1 = {t5.get('val_f1', 0):.3f} (P {t5.get('val_p', 0):.3f}, R {t5.get('val_r', 0):.3f}). "
          f"So most errors come from the verifier (label/threshold decisions on retrieved abstracts), not from "
          f"retrieval, which already reaches Recall@100 = 1.0.")

    Q5 = ("Fine-tune the NLI verifier on the SciFact training rationales (claim, gold rationale sentences → "
          "SUPPORT/CONTRADICT, plus retrieved non-rationale sentences as NEI) instead of using it zero-shot. "
          "Retrieval is already strong (nDCG@10 > 0.9, Recall@100 = 1.0), so the remaining errors are mostly "
          "verification: MNLI-style models miss SciFact-specific contradictions (opposite effect direction, "
          "negated findings, numbers) and are poorly calibrated about when an abstract is relevant but neutral. "
          "In-domain supervision targets exactly these failure modes.")

    out = ["# Report (all analysis on validation claims)", ""]
    for i, t in enumerate([Q1, Q2, Q3, Q4, Q5], 1):
        out += [f"## Q{i}", "", t, ""]
        print(f"Q{i}: {len(t.split())} words")
    with open(os.path.join(ROOT, "report.md"), "w", encoding="utf-8") as f:
        f.write("\n".join(out))


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/submission/code/requirements.txt
numpy
scipy
scikit-learn
torch
transformers>=4.40
sentence-transformers>=3.0
PyStemmer
tqdm
sentencepiece
protobuf


In [ ]:
%%writefile /content/submission/code/README.md
# SciFact evidence retrieval + claim verification — reproduction

All scripts live in `code/` and are run **from inside `code/`**. They expect the provided data in
`../../data/` (i.e. the `data/` folder next to `check_format.py`); override with the `DATA_DIR`
environment variable. Global seed = 42 (`common.SEED`). Hardware used: 1x RTX 4050 Laptop (6 GB),
Python 3.13, torch 2.13 + CUDA. Intermediate score matrices / runs go to `../cache/` (not shipped).

```bash
pip install -r requirements.txt
```

Run the tasks in order (each one reuses the cached runs of the previous ones and writes its numbers
into `../results.json` and `val_metrics.json`):

| Task | Command | Output files (`../predictions/`) | Approx. time (GPU) |
|---|---|---|---|
| 1 BM25 | `python task1_bm25.py` | `t1_bm25.trec`, `val_bm25.trec` | < 1 min |
| 2 Dense | `python task2_dense.py` | `t2_general.trec`, `t2_scientific.trec`, `val_general.trec` | ~5 min |
| 3 Hybrid + rerank | `python task3_hybrid_rerank.py` | `t3_hybrid.trec`, `t3_rerank.trec` | ~15 min |
| 4 Fine-tuning | `python task4_finetune.py` | `t4_random.trec`, `t4_hard.trec` | ~6 min |
| 5 Verification | `python task5_verify.py` | `t5_verification.csv`, `t5_val_verification.csv`, `t5_val_errors.csv` | ~5 min |
| Q1 analysis | `python report_analysis.py` | fills `report_q1` in `results.json`, prints the ablation | seconds |
| Report draft | `python make_report.py` | `../report.md` filled from the measured numbers | seconds |

Then validate: `python ../../check_format.py ..`

The submitted `report.md` starts from the `make_report.py` draft, with Q1's explanations written by hand from
`report_analysis.py`'s output. `colab_pipeline.ipynb` holds the same code in one notebook: upload the data zip, then
Runtime → Run all.

## Validation results (seed 42)

| Run | nDCG@10 | Recall@100 | MRR |
|---|---|---|---|
| t1 BM25 (k1=0.9, b=0.5) | 0.855 | 0.981 | 0.828 |
| t2 general (BGE-base) | 0.891 | 1.000 | 0.879 |
| t2 scientific (MedCPT) | 0.851 | 1.000 | 0.813 |
| t3 hybrid (weighted RRF) | 0.914 | 1.000 | 0.896 |
| t3 rerank (MedCPT cross-encoder, k=20) | **0.928** | 1.000 | 0.907 |
| t4 random negatives | 0.887 | 1.000 | 0.872 |
| t4 hard negatives | 0.892 | 1.000 | 0.877 |

Task 5 (top-2 reranked abstracts): validation P = 0.621, R = 0.476, F1 = 0.539.

## What each script does

* `common.py` – data loading, TREC writing (100 unique docs per claim, non-increasing scores),
  nDCG@10 / Recall@100 / MRR computed only over claims with gold evidence (as in grading).
* `task1_bm25.py` – corpus/claim statistics, own sparse-matrix BM25 (Lucene idf) over title+abstract;
  grid search on val nDCG@10 over k1, b, stemming and stop-word removal.
* `task2_dense.py` / `encoders.py` – general model `BAAI/bge-base-en-v1.5` (with its query instruction)
  and biomedical model `ncbi/MedCPT` (query encoder + article encoder). Exact inner-product search.
  Index time = encoding the 5,183 abstracts; query latency = encode one claim + search + top-100, mean
  over the 162 val claims, batch size 1; memory = model weights (fp16 on GPU) + fp32 flat index.
* `task3_hybrid_rerank.py` – fuses full-corpus scores of BM25, BGE and MedCPT (z-score, min-max or
  RRF). Fusion weights are grid-searched on the **train** claims (all three models are pretrained only,
  so train is unseen by them); normalisation and reranker depth k are chosen on val nDCG@10.
  The best first-stage system (by val) is reranked with a cross-encoder. `BAAI/bge-reranker-base`,
  `cross-encoder/ms-marco-MiniLM-L-12-v2` and `ncbi/MedCPT-Cross-Encoder` are compared on val with
  k ∈ {5, 10, 20, 30, 50, 100}. The top-k go in reranker order; ranks k+1..100 keep first-stage order.
* `task4_finetune.py` – fine-tunes `bge-base-en-v1.5` on the train claims (claim → gold abstract) with
  MultipleNegativesRankingLoss, (a) one random negative per pair, (b) one hard negative per pair mined
  from the top-30 of our BM25 / BGE / MedCPT train runs. lr 1e-5, batch 16, 2 epochs. A first run at
  lr 2e-5 overfit after one epoch. Val and train nDCG@10 are logged every epoch
  (`../cache/ft_*_history.json`), and the best fine-tuned epoch on val is kept.
* `task5_verify.py` – takes the top abstracts of the retriever with the best val nDCG@10, runs zero-shot
  NLI (`MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli`) per abstract sentence and on the whole abstract,
  aggregates the probabilities with a 3-way logistic regression trained on the train claims
  (decision margin + number of abstracts tuned with 5-fold out-of-fold predictions on train), and writes
  the eval and val predictions. Every val claim whose predicted (doc, label) set differs from gold is
  tagged with one error category by deterministic rules (retrieval_miss → needs_multiple_docs →
  negation / numerical cue words → entity_mismatch for wrong extra abstracts → other).
* `report_analysis.py` – per-claim val nDCG@10 BM25 vs BGE for Q1 and the ablation table.

All models are open-source encoders with ≤ 400M parameters; no generative LLMs are used.


## Provided helper files

In [ ]:
%%writefile /content/results_template.json
{
  "name": "",
  "roll_no": "",
  "seed": 0,
  "gpu_used": "",
  "task1": {
    "bm25_settings": "",
    "val_ndcg10": null
  },
  "task2": {
    "general_model": "",
    "scientific_model": "",
    "val_ndcg10_general": null,
    "val_ndcg10_scientific": null,
    "index_time_sec": {"general": null, "scientific": null},
    "query_latency_ms": {"general": null, "scientific": null},
    "memory_mb": {"general": null, "scientific": null}
  },
  "task3": {
    "hybrid_method": "",
    "reranker_model": "",
    "rerank_top_k": null,
    "val_ndcg10_hybrid": null,
    "val_ndcg10_rerank": null,
    "rerank_query_latency_ms": null
  },
  "task4": {
    "base_model": "",
    "val_ndcg10_random_negatives": null,
    "val_ndcg10_hard_negatives": null
  },
  "task5": {
    "retriever_used": "",
    "nli_model": "",
    "zero_shot_or_finetuned": "",
    "val_f1": null
  },
  "report_q1": {
    "bm25_wins": [null, null],
    "general_wins": [null, null]
  }
}


In [ ]:
%%writefile /content/check_format.py
"""Check that your NLP submission has the right files and formats BEFORE you submit.

Usage (run from this folder, so it can find data/):
    python check_format.py path/to/<rollno>_nlp_bootcamp.zip
    python check_format.py path/to/your_submission_folder

It only checks names and formats. It does NOT score your evaluation predictions.
We use the same checks when grading, so a file that fails here gets 0 marks.
"""
import csv
import json
import os
import shutil
import sys
import tempfile
import zipfile

HERE = os.path.dirname(os.path.abspath(__file__))
DATA_DIR = os.path.join(HERE, "data")

EVAL_RUNS = ["t1_bm25.trec", "t2_general.trec", "t2_scientific.trec", "t3_hybrid.trec",
             "t3_rerank.trec", "t4_random.trec", "t4_hard.trec"]
VAL_RUNS = ["val_bm25.trec", "val_general.trec"]
TASKS = {
    "task1": ["t1_bm25.trec"],
    "task2": ["t2_general.trec", "t2_scientific.trec"],
    "task3": ["t3_hybrid.trec", "t3_rerank.trec"],
    "task4": ["t4_random.trec", "t4_hard.trec"],
    "task5": ["t5_verification.csv", "t5_val_verification.csv", "t5_val_errors.csv"],
}
LABELS = {"SUPPORT", "CONTRADICT"}
CATEGORIES = {"retrieval_miss", "negation", "numerical", "entity_mismatch", "needs_multiple_docs", "other"}
TOP_K = 100


# ---------------------------------------------------------------- helpers
def locate_root(path):
    """Return (root_folder, temp_dir_or_None). root_folder contains predictions/."""
    tmp = None
    if os.path.isfile(path) and path.lower().endswith(".zip"):
        tmp = tempfile.mkdtemp()
        with zipfile.ZipFile(path) as z:
            z.extractall(tmp)
        path = tmp
    for dirpath, dirnames, _ in os.walk(path):
        if "__MACOSX" in dirpath:
            continue
        if "predictions" in dirnames:
            return dirpath, tmp
    return None, tmp


def read_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]


def read_csv(path):
    with open(path, newline="", encoding="utf-8-sig") as f:
        rows = [[c.strip() for c in r] for r in csv.reader(f) if any(c.strip() for c in r)]
    if not rows:
        raise ValueError("file is empty")
    return [h.lower() for h in rows[0]], rows[1:]


def load_data_info(data_dir=DATA_DIR):
    return {
        "doc_ids": {int(d["doc_id"]) for d in read_jsonl(os.path.join(data_dir, "corpus.jsonl"))},
        "eval_ids": {int(c["id"]) for c in read_jsonl(os.path.join(data_dir, "eval_claims.jsonl"))},
        "val_claims": read_jsonl(os.path.join(data_dir, "val_claims.jsonl")),
    }


# ---------------------------------------------------------------- per-file checks
def check_trec(path, claim_ids, doc_ids):
    """Returns {claim_id: [doc_id ranked 1..100]}"""
    runs = {}
    with open(path, encoding="utf-8") as f:
        for ln, line in enumerate(f, 1):
            if not line.strip():
                continue
            p = line.split()
            if len(p) != 6:
                raise ValueError(f"line {ln}: need 6 fields 'claim_id Q0 doc_id rank score run_name'")
            qid, did, rank = int(p[0]), int(p[2]), int(p[3])
            float(p[4])
            if qid not in claim_ids:
                raise ValueError(f"line {ln}: unknown claim_id {qid}")
            if did not in doc_ids:
                raise ValueError(f"line {ln}: unknown doc_id {did}")
            runs.setdefault(qid, {})
            if rank in runs[qid]:
                raise ValueError(f"line {ln}: claim {qid} has rank {rank} twice")
            runs[qid][rank] = did
    missing = claim_ids - set(runs)
    if missing:
        raise ValueError(f"{len(missing)} claims missing (e.g. {sorted(missing)[:3]})")
    out = {}
    for qid, r in runs.items():
        if sorted(r) != list(range(1, TOP_K + 1)):
            raise ValueError(f"claim {qid}: ranks must be exactly 1..{TOP_K}")
        docs = [r[k] for k in range(1, TOP_K + 1)]
        if len(set(docs)) != TOP_K:
            raise ValueError(f"claim {qid}: doc_ids must be unique")
        out[qid] = docs
    return out


def check_verification(path, claim_ids, doc_ids):
    """Returns set of (claim_id, doc_id, label)"""
    header, rows = read_csv(path)
    if header != ["claim_id", "doc_id", "label"]:
        raise ValueError("header must be 'claim_id,doc_id,label'")
    triples, per_claim, pairs = set(), {}, set()
    for i, r in enumerate(rows, 2):
        cid, did, lab = int(r[0]), int(r[1]), r[2].upper()
        if cid not in claim_ids:
            raise ValueError(f"row {i}: unknown claim_id {cid}")
        if did not in doc_ids:
            raise ValueError(f"row {i}: unknown doc_id {did}")
        if lab not in LABELS:
            raise ValueError(f"row {i}: label must be SUPPORT or CONTRADICT")
        if (cid, did) in pairs:
            raise ValueError(f"row {i}: (claim {cid}, doc {did}) appears twice")
        pairs.add((cid, did))
        per_claim[cid] = per_claim.get(cid, 0) + 1
        if per_claim[cid] > 3:
            raise ValueError(f"claim {cid}: more than 3 rows")
        triples.add((cid, did, lab))
    return triples


def check_errors(path, claim_ids):
    """Returns {claim_id: category}"""
    header, rows = read_csv(path)
    if header != ["claim_id", "category"]:
        raise ValueError("header must be 'claim_id,category'")
    out = {}
    for i, r in enumerate(rows, 2):
        cid, cat = int(r[0]), r[1].lower()
        if cid not in claim_ids:
            raise ValueError(f"row {i}: claim {cid} is not a validation claim")
        if cat not in CATEGORIES:
            raise ValueError(f"row {i}: category must be one of {sorted(CATEGORIES)}")
        if cid in out:
            raise ValueError(f"row {i}: claim {cid} listed twice")
        out[cid] = cat
    return out


# ---------------------------------------------------------------- whole submission
def check_submission(root, info):
    pred_dir = os.path.join(root, "predictions")
    present = set(os.listdir(pred_dir)) if os.path.isdir(pred_dir) else set()
    val_ids = {int(c["id"]) for c in info["val_claims"]}
    status, parsed, general = {}, {}, []

    def run(name, fn):
        if name not in present:
            status[name] = "MISSING"
            return
        try:
            parsed[name] = fn(os.path.join(pred_dir, name))
            status[name] = "OK"
        except Exception as e:  # noqa: BLE001
            status[name] = f"INVALID: {e}"

    for name in EVAL_RUNS:
        run(name, lambda p: check_trec(p, info["eval_ids"], info["doc_ids"]))
    for name in VAL_RUNS:
        run(name, lambda p: check_trec(p, val_ids, info["doc_ids"]))
    run("t5_verification.csv", lambda p: check_verification(p, info["eval_ids"], info["doc_ids"]))
    run("t5_val_verification.csv", lambda p: check_verification(p, val_ids, info["doc_ids"]))
    run("t5_val_errors.csv", lambda p: check_errors(p, val_ids))

    completion = {t: sum(status.get(f) == "OK" for f in fs) / len(fs) for t, fs in TASKS.items()}

    extra = sorted(present - set(status))
    if extra:
        general.append(f"Unexpected files in predictions/ (ignored): {', '.join(extra)}")

    results = None
    rj = os.path.join(root, "results.json")
    if not os.path.isfile(rj):
        general.append("results.json is MISSING")
    else:
        try:
            with open(rj, encoding="utf-8") as f:
                results = json.load(f)
            q1 = results.get("report_q1", {})
            for key in ["bm25_wins", "general_wins"]:
                ids = q1.get(key)
                if not (isinstance(ids, list) and len(ids) == 2 and all(isinstance(x, int) for x in ids)):
                    general.append(f"results.json report_q1.{key} must be a list of 2 validation claim ids")
                elif not set(ids) <= val_ids:
                    general.append(f"results.json report_q1.{key} contains ids that are not validation claims")
        except Exception as e:  # noqa: BLE001
            general.append(f"results.json is not valid JSON: {e}")
    if not os.path.isfile(os.path.join(root, "report.md")):
        general.append("report.md is MISSING")
    if not os.path.isfile(os.path.join(root, "code", "README.md")):
        general.append("code/README.md is MISSING (must say how to reproduce each task)")

    return {"status": status, "parsed": parsed, "completion": completion,
            "general": general, "results": results}


def main():
    if len(sys.argv) != 2:
        print(__doc__)
        sys.exit(1)
    root, tmp = locate_root(sys.argv[1])
    try:
        if root is None:
            print("ERROR: could not find a 'predictions/' folder in your submission.")
            sys.exit(1)
        res = check_submission(root, load_data_info())
        print("\nFile check")
        print("-" * 60)
        for name in EVAL_RUNS + VAL_RUNS + ["t5_verification.csv", "t5_val_verification.csv", "t5_val_errors.csv"]:
            print(f"  {name:26s} {res['status'][name]}")
        print("\nTask completeness (1.0 = all files valid)")
        print("-" * 60)
        for t, v in res["completion"].items():
            print(f"  {t}: {v:.2f}")
        if res["general"]:
            print("\nOther issues")
            print("-" * 60)
            for m in res["general"]:
                print("  - " + m)
        ok = all(v == 1.0 for v in res["completion"].values()) and not res["general"] \
            and all(res["status"][f] == "OK" for f in VAL_RUNS)
        print("\n" + ("All good." if ok else "Fix the issues above (or submit what you have)."))
    finally:
        if tmp:
            shutil.rmtree(tmp, ignore_errors=True)


if __name__ == "__main__":
    main()

In [ ]:
import json, torch
res = json.load(open('/content/results_template.json'))
res.update(name=NAME, roll_no=ROLL_NO, seed=42,
           gpu_used=torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none (CPU)')
json.dump(res, open('/content/submission/results.json', 'w'), indent=2)

## Run the tasks

In [ ]:
%cd /content/submission/code
!python task1_bm25.py

In [ ]:
!python task2_dense.py

In [ ]:
!python task3_hybrid_rerank.py

In [ ]:
!python task4_finetune.py

In [ ]:
!python task5_verify.py

In [ ]:
!python report_analysis.py
!python make_report.py
!cat ../report.md

## Check format and download the zip

In [ ]:
%cd /content
import shutil, os
shutil.rmtree('/content/pkg', ignore_errors=True)
shutil.copytree('/content/submission', '/content/pkg', ignore=shutil.ignore_patterns('cache', '__pycache__', '*.ipynb'))
zip_path = shutil.make_archive(f'/content/{ROLL_NO}_nlp_bootcamp', 'zip', '/content/pkg')
!python check_format.py {zip_path}
from google.colab import files
files.download(zip_path)